# 03 - Camada Gold
Dimensões, fato e tabelas-ponte a partir da Silver, mais as perguntas de negócio

In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

def salvar(df, tabela):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"OK: {tabela} -> {spark.table(tabela).count()} linhas")

def add_sk(df, nome_sk, ordem):
    w = Window.orderBy(*ordem)
    return df.withColumn(nome_sk, F.row_number().over(w).cast("bigint")) \
             .select(nome_sk, *[c for c in df.columns])

## Dimensões

In [0]:
# dim_movies
info = spark.table("silver.tb_info_filmes").select(
    F.col("id_filme").cast("string"), "titulo", "data_lancamento",
    F.col("ano_lancamento").cast("int"), F.col("duracao_minutos").cast("int"),
    "idioma_original", "status_filme", "sinopse")
dim_movies = add_sk(info, "sk_movie_id", ["id_filme"])
salvar(dim_movies, "gold.dim_movies")

# dim_genres
dim_genres = add_sk(
    spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct(),
    "sk_genre_id", ["nome_genero"])
salvar(dim_genres, "gold.dim_genres")

# dim_people  e dim_companies
pe = spark.table("silver.tb_pessoas_empresas")
dim_people = add_sk(
    pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
      .select(F.col("nome_entidade").alias("nome_pessoa"),
              F.col("tipo_entidade").alias("tipo_pessoa")).distinct(),
    "sk_person_id", ["nome_pessoa", "tipo_pessoa"])
salvar(dim_people, "gold.dim_people")

dim_companies = add_sk(
    pe.filter(F.col("tipo_entidade") == "Produtora")
      .select(F.col("nome_entidade").alias("nome_produtora")).distinct(),
    "sk_company_id", ["nome_produtora"])
salvar(dim_companies, "gold.dim_companies")

# dim_reviews 
mv = dim_movies.select("sk_movie_id", "id_filme")
resumo = (spark.table("silver.tb_avaliacoes_usuarios").groupBy("id_filme")
          .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
               F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"))
          .join(mv, "id_filme"))      # inner: só filmes que existem na dim_movies
dim_reviews = add_sk(resumo.select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"),
                     "sk_review_id", ["sk_movie_id"])
salvar(dim_reviews, "gold.dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: gold.dim_movies -> 97611 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: gold.dim_genres -> 19 linhas
OK: gold.dim_people -> 418536 linhas
OK: gold.dim_companies -> 45863 linhas
OK: gold.dim_reviews -> 27226 linhas


grão: 1 linha por filme lançado

In [0]:
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

# Regra de negócio: a fato consolida apenas filmes com status 'Lançado'. Filmes em produção, planejados ou rumores ainda não têm receita nem votos reais, então distorceriam as métricas. As tabelas Silver são únicas por id_filme, então os LEFT JOINs abaixo não duplicam o grão. LEFT para manter o filme mesmo sem dado financeiro/engajamento.
fact = (dim_movies.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
        .join(fin, "id_filme", "left").join(met, "id_filme", "left")
        .select("sk_movie_id",
                F.col("orcamento_usd").cast("decimal(18,2)"), F.col("receita_usd").cast("decimal(18,2)"),
                F.col("lucro_usd").cast("decimal(18,2)"),
                F.col("orcamento_brl").cast("decimal(18,2)"), F.col("receita_brl").cast("decimal(18,2)"),
                F.col("lucro_brl").cast("decimal(18,2)"),
                F.col("popularidade").cast("double"), F.col("nota_media_tmdb").cast("double"),
                F.col("qtd_votos_tmdb").cast("int"), F.col("nota_media_imdb").cast("double"),
                F.col("qtd_votos_imdb").cast("int")))

assert fact.count() == fact.select("sk_movie_id").distinct().count(), "Grão duplicado na fato!"
salvar(fact, "gold.fact_movies_performance")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: gold.fact_movies_performance -> 96261 linhas


## Tabelas-ponte (bridges)

In [0]:
b_genre = (spark.table("silver.tb_generos").join(mv, "id_filme")
           .join(dim_genres, F.col("genero") == F.col("nome_genero"))
           .select("sk_movie_id", "sk_genre_id").distinct())
salvar(b_genre, "gold.bridge_movie_genre")

b_person = (pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
            .join(mv, "id_filme")
            .join(dim_people, (pe["nome_entidade"] == dim_people["nome_pessoa"]) &
                              (pe["tipo_entidade"] == dim_people["tipo_pessoa"]))
            .select("sk_movie_id", "sk_person_id").distinct())
salvar(b_person, "gold.bridge_movie_person")

b_comp = (pe.filter(F.col("tipo_entidade") == "Produtora").join(mv, "id_filme")
          .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"))
          .select("sk_movie_id", "sk_company_id").distinct())
salvar(b_comp, "gold.bridge_movie_company")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: gold.bridge_movie_genre -> 139858 linhas
OK: gold.bridge_movie_person -> 773271 linhas
OK: gold.bridge_movie_company -> 118776 linhas


## Desafio de Analytics

### 1. Receita total (em R$) de todos os filmes da base

In [0]:
# Receita total em R$
display(spark.table("gold.fact_movies_performance")
        .agg(F.sum("receita_brl").alias("receita_total_brl")))

receita_total_brl
808185213454.95


### 2. Os 5 filmes com maior popularidade

In [0]:
# Top 5 popularidade
display(spark.table("gold.fact_movies_performance")
        .join(spark.table("gold.dim_movies"), "sk_movie_id")
        .select("titulo", "popularidade")
        .orderBy(F.col("popularidade").desc_nulls_last()).limit(5))

titulo,popularidade
The Greatest Show on Turf,5003.0
Purge of Kingdoms,3000.0
blue beetle,2994.357
Gran Turismo,2680.593
Obamaland Part 1: Rise of the Trumpublikans,2040.0


### 3. Quantidade de filmes por gênero (maior para menor)

In [0]:
# Filmes por gênero
display(spark.table("gold.bridge_movie_genre")
        .join(spark.table("gold.dim_genres"), "sk_genre_id")
        .groupBy("nome_genero").agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
        .orderBy(F.col("qtd_filmes").desc()))

nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18535
Thriller,10242
Horror,9674
Romance,7618
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


### 4. Top 10 filmes por receita (US$ e R$) com RANK()

In [0]:
# Top 10 receita com RANK
w_rank = Window.orderBy(F.col("receita_usd").desc())
display(spark.table("gold.fact_movies_performance")
        .filter(F.col("receita_usd").isNotNull())
        .join(spark.table("gold.dim_movies"), "sk_movie_id")
        .withColumn("ranking", F.rank().over(w_rank))
        .filter(F.col("ranking") <= 10)
        .select("ranking", "titulo", "receita_usd", "receita_brl").orderBy("ranking"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13980120000.00
2,Avatar: The Way of Water,2320250281.00,11584777628.00
3,AVENGERS: INFINITY WAR,2052415039.00,10247503048.22
4,spider-man: no way home,1921847111.00,9595590440.51
5,The Lion King,1663075401.00,8303569169.65
6,Top Gun: Maverick,1488732821.00,7433094101.97
7,Barbie,1428545028.00,7132582470.30
8,The Super Mario Bros. Movie,1355725263.00,6769000665.63
9,Black Panther,1349926083.00,6740045939.81
10,Star Wars: The Last Jedi,1332698830.00,6654031988.31


### 5. Ator com mais participações nos últimos 2 anos

In [0]:
# Data de referência: lançamento mais recente, ignorando datas futuras e filmes não lançados
dm = spark.table("gold.dim_movies")
ref = (dm.filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
         .agg(F.max("data_lancamento")).first()[0])
print("Data de referência:", ref)

def janela(anos):
    return dm.filter((F.col("status_filme") == "Lançado") &
                     (F.col("data_lancamento") <= F.lit(ref)) &
                     (F.col("data_lancamento") > F.add_months(F.lit(ref), -12 * anos)))

# 5) Ator com mais participações nos últimos 2 anos
atores = (janela(2)
          .join(spark.table("gold.bridge_movie_person"), "sk_movie_id")
          .join(spark.table("gold.dim_people").filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
          .groupBy("nome_pessoa").agg(F.countDistinct("sk_movie_id").alias("qtd_filmes")))

# dense_rank = 1 traz TODOS os atores empatados na 1ª posição (hoje há empate)
w_at = Window.orderBy(F.col("qtd_filmes").desc())
display(atores.withColumn("posicao", F.dense_rank().over(w_at))
              .filter("posicao = 1").drop("posicao")
              .orderBy("nome_nome" if False else "nome_pessoa"))

Data de referência: 2026-02-19


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_pessoa,qtd_filmes
Adlih Torres,12
Andy Dubitsky,12
Cooper Tomlinson,12
Gloria Karel,12
John Simmonds,12
Jonnathon Cripple,12
Sterling L. Pope,12
Tristan Welsh,12


### 6. Produtora com maior lucro nos últimos 5 anos

In [0]:
# Produtora com maior lucro nos últimos 5 anos
display(janela(5)
        .join(spark.table("gold.fact_movies_performance"), "sk_movie_id")
        .filter(F.col("lucro_usd").isNotNull())
        .join(spark.table("gold.bridge_movie_company"), "sk_movie_id")
        .join(spark.table("gold.dim_companies"), "sk_company_id")
        .groupBy("nome_produtora")
        .agg(F.sum("lucro_usd").alias("lucro_total_usd"), F.sum("lucro_brl").alias("lucro_total_brl"))
        .orderBy(F.col("lucro_total_usd").desc()).limit(5))

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5772329679.00,28820664854.26
Marvel Studios,4953462823.00,24732144528.96
Columbia Pictures,3662050755.00,18284253214.65
Pascal Pictures,2701952454.00,13490578407.58
Illumination,2431353473.00,12139504755.34
